In [ ]:
! pip install otter-grader

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("tarea1_alumno.ipynb")

# EO4040 — Tarea 1: Calidad y limpieza de datos
## Dataset: ENIGH 2022 — concentradohogar

**Valor total:** 80 pts automáticos + 20 pts presentación en clase

---

### Instrucciones generales

1. Descarga el archivo `concentradohogar.csv` desde: https://www.inegi.org.mx/programas/enigh/nc/2022/
2. Coloca el archivo en la **misma carpeta que este notebook**
3. Corre todas las celdas **en orden, de arriba a abajo**, sin saltar ninguna
4. **Los nombres de las variables importan.** Cada pregunta indica el nombre exacto que debe tener tu variable — si la llamas diferente, el calificador automático no la encontrará y la pregunta quedará en cero
5. El dataset original (`df`) **nunca debe modificarse** — toda limpieza se hace sobre copias
6. Al terminar, exporta tu entrega con la celda al final del notebook

---

In [ ]:
# Celda de configuración — no modificar
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print('✅ Librerías cargadas correctamente')

---
## Parte 1 — Cargar el dataset
**10 puntos**

### Pregunta 1 — Cargar el archivo (10 pts)

Carga el archivo `concentradohogar.csv` en un DataFrame.

**Requisitos:**
- Usa `encoding='latin-1'` para evitar errores con caracteres especiales del español
- El DataFrame debe llamarse exactamente **`df`**

El calificador verifica que:
- `df` exista
- Tenga al menos 70,000 filas (la ENIGH 2022 tiene ~89,000 hogares)
- Tenga al menos 100 columnas
- Contenga la columna `folioviv` (identificador del hogar)

In [ ]:
df = ...
print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')

In [ ]:
grader.check("q1")

---
## Parte 2 — Diagnóstico de calidad
**20 puntos**

### Pregunta 2 — Tabla de calidad (10 pts)

Construye un resumen de calidad del dataset.

**La variable debe llamarse exactamente `tabla_calidad`** y ser un DataFrame con estas 4 columnas en este orden:

| Columna | Contenido |
|---|---|
| `variable` | Nombre de cada columna del dataset |
| `tipo_dato` | Tipo de dato en Python — usa `str(df[col].dtype)` |
| `pct_faltantes` | Porcentaje de valores faltantes (0.0 a 100.0), redondeado a 2 decimales |
| `n_unicos` | Número de valores únicos sin contar NaN — usa `.nunique()` |

Debe tener una fila por cada columna del dataset original (todas, no solo algunas).

In [ ]:
...
tabla_calidad.head(10)

In [ ]:
grader.check("q2")

### Pregunta 3 — Identificar problemas (10 pts)

Con base en tu `tabla_calidad`, crea las siguientes tres variables con **exactamente estos nombres**:

| Variable | Tipo | Contenido |
|---|---|---|
| `vars_con_faltantes` | `list` | Nombres de columnas con más de 0% de faltantes |
| `n_duplicados` | `int` | Número de filas con `folioviv` duplicado |
| `col_mas_faltantes` | `str` | Nombre de la columna con el mayor porcentaje de faltantes |

In [ ]:
...
print(f'Variables con faltantes: {len(vars_con_faltantes)}')
print(f'Duplicados en folioviv:  {n_duplicados}')
print(f'Columna con más faltantes: {col_mas_faltantes}')

In [ ]:
grader.check("q3")

---
## Parte 3 — Limpieza
**30 puntos**

### Pregunta 4 — Crear copia y eliminar duplicados (10 pts)

**La variable debe llamarse exactamente `df_clean`.**

1. Crea una copia del dataset original con `.copy()` y guárdala en `df_clean`
2. Sobre `df_clean`, elimina los registros con `folioviv` duplicado conservando la primera aparición

El dataset original `df` **no debe modificarse** — el calificador lo verifica.

> **¿Por qué eliminar y no conservar ambos?** En la ENIGH, `folioviv` identifica de forma única a cada hogar. Un duplicado es un error de captura o de procesamiento, no un hogar real distinto.

In [ ]:
...
print(f'Filas originales:                  {len(df):,}')
print(f'Filas después de quitar duplicados: {len(df_clean):,}')

In [ ]:
grader.check("q4")

### Pregunta 5 — Imputar ingreso corriente (10 pts)

La variable `ingcor` es el ingreso corriente trimestral del hogar. Tiene valores faltantes.

Sobre `df_clean`, realiza los siguientes dos pasos **en este orden**:

1. Crea la columna **`ingcor_faltante`**: vale `1` si `ingcor` era NaN antes de imputar, `0` si no
2. Imputa los NaN de **`ingcor`** con la mediana de la columna

> **¿Por qué la mediana y no la media?** La distribución del ingreso en México es muy asimétrica — hay hogares con ingresos extremadamente altos que jalan la media hacia arriba. La mediana es más representativa del hogar típico y no se ve afectada por esos valores extremos.

In [ ]:
...
print(f'Valores faltantes en ingcor después de imputar: {df_clean["ingcor"].isnull().sum()}')
print(f'Hogares con ingreso imputado (ingcor_faltante=1): {df_clean["ingcor_faltante"].sum()}')

In [ ]:
grader.check("q5")

### Pregunta 6 — Corregir el tipo de `est_socio` (10 pts)

La variable `est_socio` indica el estrato socioeconómico del hogar:
- 1 = Bajo
- 2 = Medio bajo
- 3 = Medio alto
- 4 = Alto

Tiene un orden natural — es una variable **ordinal**, no solo nominal. Pandas no lo sabe a menos que se lo indiquemos.

Convierte `df_clean['est_socio']` a tipo categórico ordenado usando:
```python
pd.CategoricalDtype(categories=[1, 2, 3, 4], ordered=True)
```

> **¿Por qué importa?** Con `ordered=True`, Pandas puede hacer comparaciones como `est_socio > 2` y los modelos de ML pueden usar correctamente el orden de los niveles.

In [ ]:
...
print(f'Tipo de est_socio: {df_clean["est_socio"].dtype}')
print(f'¿Ordenada?: {df_clean["est_socio"].cat.ordered}')
print(df_clean['est_socio'].value_counts().sort_index())

In [ ]:
grader.check("q6")

---
## Parte 4 — Verificación
**20 puntos**

### Pregunta 7 — Reporte comparativo (10 pts)

Crea un DataFrame llamado **`reporte`** que compare el dataset original con el limpio.

Debe tener exactamente estas 3 columnas y 2 filas:

| `dataset` | `n_filas` | `total_nan` |
|---|---|---|
| `'original'` | número de filas de `df` | suma de todos los NaN en `df` |
| `'limpio'` | número de filas de `df_clean` | suma de todos los NaN en `df_clean` |

Para calcular el total de NaN en un DataFrame: `df.isnull().sum().sum()`

In [ ]:
...
reporte

In [ ]:
grader.check("q7")

### Pregunta 8 — Guardar el dataset limpio (10 pts)

1. Guarda `df_clean` en un archivo CSV llamado **`enigh2022_limpio.csv`** sin el índice de Pandas
2. Vuelve a cargarlo en una variable llamada **`df_verificacion`**
3. El calificador verifica que el archivo existe y que `df_verificacion` tiene el mismo número de filas que `df_clean`

In [ ]:
...
print(f'Archivo guardado: {df_verificacion.shape[0]:,} filas × {df_verificacion.shape[1]} columnas')

In [ ]:
grader.check("q8")

---
## Presentación en clase (20 pts)

Esta parte no se califica aquí — la presentarás en clase en **5 minutos**.

Prepárate para responder cualquiera de estas preguntas sin ver el notebook:

1. ¿Cuántas variables tenían faltantes? ¿Cuál tenía más? ¿Por qué crees que falta ese dato?
2. ¿Por qué usaste la mediana para imputar `ingcor` y no la media?
3. Si los hogares con ingreso muy alto o muy bajo son los que más faltan en `ingcor` (lo que se llama MNAR), ¿qué sesgo introduce tu imputación con mediana?
4. ¿Cómo afectaría ese sesgo a un análisis de desigualdad de ingresos en México?

---
## Exportar tu entrega

Corre la siguiente celda para generar el archivo `.zip` que subirás a Canvas.

---

Para comprobar tu trabajo, la celda de abajo volverá a ejecutar todas las pruebas del autocalificador.

In [ ]:
grader.check_all()

## Entrega

Asegúrate de haber ejecutado todas las celdas de tu notebook en orden antes de ejecutar la celda de abajo, para que todas las imágenes y gráficos aparezcan en la salida. La celda siguiente generará un archivo ZIP para que lo entregues. **¡Guarda tu trabajo antes de exportar!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True)